# 03-7 모듈과 패키지 실습

import가 이름을 만드는 방식, 모듈 캐시와 검색 경로, 일반 패키지, 상대 import, `python -m` 실행을 확인합니다. 임시 디렉터리에 작은 패키지를 만들고 마지막에 정리하므로 저장소 파일은 변경하지 않습니다.

## 1. import가 만드는 이름

In [ ]:
import math
import statistics as stats
from pathlib import Path

assert math.sqrt(16) == 4.0
assert stats.mean([10, 20, 30]) == 20
assert Path("report.txt").suffix == ".txt"
assert math.__name__ == "math"
print("현재 셀의 주요 이름:", ["math", "stats", "Path"])

## 2. 모듈 객체와 import 캐시

In [ ]:
import json
import json as json_again
import sys

assert json is json_again
assert sys.modules["json"] is json
assert "loads" in dir(json)
print("모듈 이름:", json.__name__)
print("모듈 위치:", getattr(json, "__file__", None))

## 3. 검색 경로와 모듈 존재 확인

In [ ]:
from importlib.util import find_spec

assert find_spec("json") is not None
assert find_spec("module_that_does_not_exist_12345") is None
print("Python 실행 파일:", sys.executable)
print("현재 작업 디렉터리:", Path.cwd())
print("검색 경로 앞부분:", sys.path[:3])

## 3.1 이후 과정의 모듈·패키지 지도

표준 모듈은 현재 Python에서 발견되는지 확인하고, 외부 패키지는 설치 이름과 import 이름을 구분합니다.

| 과정 | 표준 모듈 | 외부 패키지 |
| --- | --- | --- |
| 파일·데이터 | `pathlib`, `csv`, `json`, `hashlib` | NumPy, pandas |
| 텍스트 분석 | `re`, `datetime`, `collections`, `ipaddress` | NumPy, pandas |
| 네트워크·HTTP | `socket`, `struct`, `urllib.parse` | requests |
| 자동화·테스트 | `subprocess`, `argparse`, `logging` | pytest |
| 보안 심화 | 표준 모듈과 결합 | Beautiful Soup, lxml, Scapy, pwntools, PyCryptodome |

In [ ]:
course_standard_modules = {
    "04-file-data": ["pathlib", "csv", "json", "io", "hashlib"],
    "05-text-analysis": ["re", "unicodedata", "datetime", "collections", "ipaddress"],
    "06-network": ["socket", "struct", "time"],
    "07-http": ["urllib.parse", "http.server", "ssl"],
    "08-automation": ["os", "subprocess", "argparse", "logging"],
    "09-11-structure": ["traceback", "unittest.mock", "typing", "dataclasses", "asyncio", "concurrent.futures"],
}

missing_standard = [
    module_name
    for module_names in course_standard_modules.values()
    for module_name in module_names
    if find_spec(module_name) is None
]
assert missing_standard == []

distribution_to_import = {
    "requests": "requests",
    "numpy": "numpy",
    "pandas": "pandas",
    "pytest": "pytest",
    "beautifulsoup4": "bs4",
    "scapy": "scapy",
    "pwntools": "pwn",
    "pycryptodome": "Crypto",
}
assert distribution_to_import["beautifulsoup4"] == "bs4"
assert distribution_to_import["pwntools"] == "pwn"
assert distribution_to_import["pycryptodome"] == "Crypto"
print("표준 모듈 확인:", sum(map(len, course_standard_modules.values())), "개")
print("설치명과 import명이 다른 예:", {k: v for k, v in distribution_to_import.items() if k.lower() != v.lower()})

## 4. 격리된 실습 프로젝트 만들기

`TemporaryDirectory` 안에 일반 패키지를 만듭니다. `__init__.py`가 있는 `event_tools`가 import 패키지입니다.

In [ ]:
import tempfile
import textwrap

temp_project = tempfile.TemporaryDirectory()
project_root = Path(temp_project.name)
package_root = project_root / "event_tools"
package_root.mkdir()

def write_module(relative_path, source):
    destination = project_root / relative_path
    destination.write_text(textwrap.dedent(source).lstrip(), encoding="utf-8")
    return destination

print("실습 프로젝트:", project_root)

## 5. 검증·파싱·보고 모듈 작성

In [ ]:
write_module("event_tools/validators.py", r'''
def parse_port(value):
    if type(value) not in {int, str}:
        raise TypeError("port는 정수 또는 정수 문자열이어야 합니다")
    try:
        port = int(value)
    except ValueError as exc:
        raise ValueError("port를 정수로 변환할 수 없습니다") from exc
    if not 1 <= port <= 65535:
        raise ValueError("port는 1~65535 범위여야 합니다")
    return port

def normalize_action(action):
    if not isinstance(action, str):
        raise TypeError("action은 문자열이어야 합니다")
    normalized = action.strip().upper()
    if normalized not in {"ALLOW", "DENY"}:
        raise ValueError("action은 ALLOW 또는 DENY여야 합니다")
    return normalized
''')

write_module("event_tools/parser.py", r'''
from .validators import normalize_action, parse_port

def parse_event_line(line):
    if not isinstance(line, str):
        raise TypeError("이벤트 행은 문자열이어야 합니다")
    parts = line.split()
    if len(parts) != 3:
        raise ValueError("필드 수는 3개여야 합니다")
    action_text, ip, port_text = parts
    return {
        "action": normalize_action(action_text),
        "ip": ip,
        "port": parse_port(port_text),
    }

def parse_event_lines(lines):
    events = []
    errors = []
    for line_number, line in enumerate(lines, start=1):
        try:
            event = parse_event_line(line)
        except (TypeError, ValueError) as exc:
            errors.append({
                "line": line_number,
                "type": type(exc).__name__,
                "message": str(exc),
            })
            continue
        events.append(event)
    return {"events": events, "errors": errors}
''')

write_module("event_tools/report.py", r'''
def summarize(result):
    return {
        "event_count": len(result["events"]),
        "error_count": len(result["errors"]),
        "allowed": sum(e["action"] == "ALLOW" for e in result["events"]),
        "denied": sum(e["action"] == "DENY" for e in result["events"]),
    }
''')

assert (package_root / "validators.py").exists()
assert (package_root / "parser.py").exists()
assert (package_root / "report.py").exists()

## 6. 공개 API와 패키지 진입점 작성

In [ ]:
write_module("event_tools/__init__.py", r'''
from .parser import parse_event_line, parse_event_lines
from .report import summarize

__all__ = ["parse_event_line", "parse_event_lines", "summarize"]
''')

write_module("event_tools/__main__.py", r'''
from .parser import parse_event_lines
from .report import summarize

def main():
    lines = [
        "ALLOW 10.0.0.5 443",
        "DENY 198.51.100.9 22",
        "BLOCK 203.0.113.10 80",
    ]
    print(summarize(parse_event_lines(lines)))
    return 0

raise SystemExit(main())
''')

print([path.name for path in sorted(package_root.iterdir())])

## 7. 패키지를 import해 함수 검증

In [ ]:
sys.path.insert(0, str(project_root))

import event_tools
from event_tools import parse_event_line, parse_event_lines, summarize

event = parse_event_line("ALLOW 10.0.0.5 443")
result = parse_event_lines([
    "ALLOW 10.0.0.5 443",
    "DENY 198.51.100.9 22",
    "BLOCK 203.0.113.10 80",
])
summary = summarize(result)

assert event == {"action": "ALLOW", "ip": "10.0.0.5", "port": 443}
assert summary == {"event_count": 2, "error_count": 1, "allowed": 1, "denied": 1}
assert event_tools.__name__ == "event_tools"
assert set(event_tools.__all__) == {"parse_event_line", "parse_event_lines", "summarize"}
print(summary)

## 8. `python -m event_tools` 실행

현재 노트북과 같은 Python 실행 파일을 사용하되 작업 디렉터리는 프로젝트 루트로 지정합니다.

In [ ]:
import ast
import subprocess

completed = subprocess.run(
    [sys.executable, "-m", "event_tools"],
    cwd=project_root,
    text=True,
    capture_output=True,
    check=True,
)
module_output = ast.literal_eval(completed.stdout.strip())
assert module_output == summary
assert completed.stderr == ""
print("-m 실행 결과:", module_output)

## 9. import 오류 유형 구분

In [ ]:
observed_errors = []

try:
    import module_that_does_not_exist_12345
except ModuleNotFoundError as exc:
    observed_errors.append(type(exc).__name__)
    assert exc.name == "module_that_does_not_exist_12345"

try:
    from math import function_that_does_not_exist
except ImportError as exc:
    observed_errors.append(type(exc).__name__)

try:
    math.function_that_does_not_exist()
except AttributeError as exc:
    observed_errors.append(type(exc).__name__)

assert observed_errors == ["ModuleNotFoundError", "ImportError", "AttributeError"]
print(observed_errors)

## 10. 로컬 파일의 표준 모듈 이름 가림 확인

In [ ]:
shadow_root = project_root / "shadow_demo"
shadow_root.mkdir()
(shadow_root / "json.py").write_text("ORIGIN = 'local json.py'\n", encoding="utf-8")

shadowed = subprocess.run(
    [sys.executable, "-c", "import json; print(json.__file__)"],
    cwd=shadow_root,
    text=True,
    capture_output=True,
    check=True,
)
assert Path(shadowed.stdout.strip()).name == "json.py"
print("가져온 json 위치:", shadowed.stdout.strip())
print("해결: 로컬 파일 이름을 json_helpers.py처럼 변경합니다.")

## 11. 정리와 최종 확인

In [ ]:
for module_name in list(sys.modules):
    if module_name == "event_tools" or module_name.startswith("event_tools."):
        del sys.modules[module_name]
sys.path.remove(str(project_root))
temp_project.cleanup()

assert not project_root.exists()
print("임시 프로젝트를 정리했습니다. 전체 실습 통과!")

다음 질문에 답할 수 있으면 완료입니다.

1. `import module`과 `from module import name`이 만드는 이름은 어떻게 다른가요?
2. 같은 모듈을 다시 import할 때 최상위 코드가 보통 다시 실행되지 않는 이유는 무엇인가요?
3. `sys.path`와 `module.__file__`은 어떤 import 문제를 찾는 데 도움이 되나요?
4. 상대 import가 있는 패키지를 왜 프로젝트 루트에서 `python -m`으로 실행하나요?
5. `__init__.py`와 `__main__.py`의 역할은 각각 무엇인가요?